# 01 · PyTorch in one sitting — by training Pebble's first tiny brain

**Goal:** understand tensors, autograd and the training loop by building a real (tiny) thing:
a classifier that reads a quick-add line in Hindi, Hinglish or English and guesses the **intent**
(`remind`, `water`, `note`).

No pretrained model yet — just characters → numbers → a 2-layer network. Notebook 02 swaps in a real
multilingual encoder and you'll *see* why that matters.

Run cells top to bottom with **Shift+Enter**.

In [ ]:
import torch
from torch import nn

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Training on:", device)

## 1. Tensors — n-dimensional arrays that can live on the GPU
Everything in deep learning is a tensor: inputs, weights, outputs, gradients.

In [ ]:
a = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
b = torch.ones(2, 2)
print(a @ b)            # matrix multiply
print(a.shape, a.dtype)
print(a.to(device))     # same data, now on the GPU (if you have one)

## 2. Autograd — PyTorch computes derivatives for you
Mark a tensor with `requires_grad=True`, compute something, call `.backward()`, read `.grad`.
This is how a network learns which way to nudge each weight.

In [ ]:
w = torch.tensor(3.0, requires_grad=True)
loss = (w * 2 - 10) ** 2      # smallest when w = 5
loss.backward()
print("d loss / d w at w=3:", w.grad)   # negative → increase w

## 3. A tiny dataset in three scripts
Real data comes in notebook 03 (Amazon MASSIVE + your own recordings). For now, a handful of lines
in **Devanagari, Roman Hinglish and English** — the mix Pebble must handle.

In [ ]:
data = [
    ("remind me to call mom at 7pm", "remind"),
    ("kal 5 baje DBMS assignment yaad dila dena", "remind"),
    ("शाम को दवा लेने की याद दिलाना", "remind"),
    ("remind me tomorrow about the meeting", "remind"),
    ("mujhe 10 minute baad yaad dilana", "remind"),
    ("ek glass paani pi liya", "water"),
    ("drank 2 glasses of water", "water"),
    ("मैंने पानी पी लिया", "water"),
    ("+1 water", "water"),
    ("paani piya abhi", "water"),
    ("note: idea for the pet climbing windows", "note"),
    ("yeh likh lo: grocery mein doodh lena hai", "note"),
    ("नोट करो कल की क्लास कैंसिल है", "note"),
    ("save a note about the project deadline", "note"),
    ("likh ke rakh lo ki wifi password badalna hai", "note"),
]
labels = sorted({y for _, y in data})
print(labels)

## 4. Text → numbers: hashed character n-grams
Networks only eat numbers. We hash every 1–3 character chunk into one of 512 buckets and count them.
It works for *any* script (Devanagari included) with zero vocabulary — but it has no idea that
"paani" and "water" mean the same thing. Keep that in mind.

In [ ]:
import zlib

DIM = 512

def featurize(text: str) -> torch.Tensor:
    t = f" {text.lower()} "
    v = torch.zeros(DIM)
    for n in (1, 2, 3):
        for i in range(len(t) - n + 1):
            v[zlib.crc32(t[i:i+n].encode()) % DIM] += 1
    return v / v.norm()

X = torch.stack([featurize(t) for t, _ in data]).to(device)
y = torch.tensor([labels.index(l) for _, l in data]).to(device)
X.shape, y.shape

## 5. The model and the training loop
The loop is the same for every model you'll ever train, from this toy to Whisper:
**forward → loss → backward → optimiser step**, repeated.

In [ ]:
torch.manual_seed(0)
model = nn.Sequential(nn.Linear(DIM, 64), nn.ReLU(), nn.Linear(64, len(labels))).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=1e-2)
loss_fn = nn.CrossEntropyLoss()

for step in range(200):
    logits = model(X)            # forward
    loss = loss_fn(logits, y)    # how wrong are we?
    opt.zero_grad()
    loss.backward()              # gradients for every weight
    opt.step()                   # nudge weights downhill
    if step % 40 == 0:
        acc = (logits.argmax(1) == y).float().mean().item()
        print(f"step {step:3d}  loss {loss.item():.3f}  train acc {acc:.0%}")

## 6. The honest test: sentences it has never seen
Training accuracy is easy. What matters is **new** sentences — especially ones that share meaning
but not spelling with the training data.

In [ ]:
tests = [
    ("yaad dilana ki 6 baje gym jaana hai", "remind"),
    ("I just had some water", "water"),
    ("पानी की बोतल खत्म", "water"),
    ("jot this down: buy a new charger", "note"),
    ("remind me to drink water at 4", "remind"),   # tricky: mentions water but is a reminder
]
model.eval()
with torch.no_grad():
    for text, want in tests:
        p = model(featurize(text).to(device)).softmax(-1)
        got = labels[p.argmax().item()]
        print(f"{'✓' if got == want else '✗'} {text!r:45} → {got} ({p.max().item():.0%})")

## What you just learned
* **Tensors, autograd, the loop** — the whole mechanism of training.
* **Overfitting on a tiny set** — 100% train accuracy, shaky on new sentences.
* **Representation is everything** — character hashing can't know *paani = water*.

➡️ Notebook 02 replaces `featurize` with a pretrained **multilingual encoder** (Pebble's future
common base) and the same tiny head suddenly generalises. That's transfer learning.